## Import necessary modules
Run this cell before running any other cells

In [18]:
%load_ext autoreload
%autoreload 2

from ble import get_ble_controller
from base_ble import LOG
from cmd_types import CMD
import time
import numpy as np

LOG.propagate = False

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Printing and Logging
## Printing
You can use the **print()** function in Python to print messages to the screen. <br>
The message can be a string, or any other object, the object will be converted into a string before it is written to the screen. <br>

## Logging
You could use the logging module that is setup in *utils.py*. <br>
It prints to both your screen (standard output) as well as to log files (*ble.log*) in the *logs* directory. <br>
This is the recommended way to output messages, since the log files can help with debugging. <br>
The logging module also provides different log levels as shown below, each formatted with a different color for increased visibility. <br>

__**NOTE**__: You may notice that the DEBUG message is not printed to the screen but is printed in the log file. This is because the logging level for the screen is set to INFO and for the file is set to DEBUG. You can change the default log levels in *utils.py* (**STREAM_LOG_LEVEL** and **FILE_LOG_LEVEL**). 

## Formatting output
To format your strings, you may use %-formatting, str.format() or f-strings. <br>
The most "pythonic" way would be to use f-strings. [Here](https://realpython.com/python-f-strings/) is a good tutorial on f-strings. <br>

In [2]:
LOG.debug("debug")
LOG.info("info")
LOG.warning("warning")
LOG.error("error")
LOG.critical("critical")

2026-02-04 02:34:09,279 | INFO     |: info
2026-02-04 02:34:09,280 | WARNING  |: warning
2026-02-04 02:34:09,281 | ERROR    |: error
2026-02-04 02:34:09,282 | CRITICAL |: critical


<hr>

# BLE
## ArtemisBLEController
The class **ArtemisBLEController** (defined in *ble.py*) provides member functions to handle various BLE operations to send and receive data to/from the Artemis board, provided the accompanying Arduino sketch is running on the Artemis board. <br>

<table align="left">
     <tr>
        <th style="text-align: left; font-size: medium">Member Functions</th>
        <th style="text-align: left; font-size: medium">Description</th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">reload_config()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Reload changes made in <em>connection.yaml.</em></span></th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">connect()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Connect to the Artemis board, whose MAC address is specified in <em>connection.yaml</em>.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">disconnect()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Disconnect from the Artemis board.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">is_connected()</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Return a boolean indicating whether your controller is connected to the Artemis board or not.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">send_command(cmd_type, data)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Send the command <strong>cmd_type</strong> (integer) with <strong>data</strong> (string) to the Artemis board.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">receive_float(uuid) <br> receive_string(uuid) <br> receive_int(uuid)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Read the GATT characteristic (specified by its <strong>uuid</strong>) of type float, string or int. <br> The type of the GATT
            characteristic is determined by the classes BLEFloatCharacteristic, BLECStringCharacteristic or
            BLEIntCharacteristic in the Arduino sketch.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">start_notify(uuid, notification_handler)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Activate notifications on the GATT characteristic (specified by its <strong>uuid</strong>). <br> <strong>notification_handler</strong> is a
            function callback which must accept two inputs; the first will be a uuid string object and the second will
            be the bytearray of the characteristic value.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">bytearray_to_float(byte_array) <br> bytearray_to_string(byte_array) <br> bytearray_to_int(byte_array)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Convert the <strong>bytearray</strong> to float, string or int, respectively. <br> You may use these functions inside your
            notification callback function.</span></th>
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">stop_notify(uuid)</span></th>
        <th style="text-align: left"><span style="font-weight: normal">Stop notifications on the GATT characteristic (specified by its <strong>uuid</strong>).</span></th>
    </tr>
</table>

<table align="left">
     <tr>
        <th style="text-align: left; font-size: medium">Member Variables</th>
        <th style="text-align: left; font-size: medium">Description</th style="text-align: left">
    </tr>
    <tr>
        <th style="text-align: left"><span style="color:rgb(201,152,4);font-family:monospace">uuid</span></th>
        <th style="text-align: left"><span style="font-weight: normal">A dictionary that stores the UUIDs of the various characteristics specified in <em>connection.yaml</em>.</span></th>
    </tr>
</table>

## Configuration
- The MAC address, Service UUID and GATT characteristic UUIDs are defined in the file: *connection.yaml*.
- They should match the UUIDs used in the Arduino sketch.
- The artemis board running the base code should display its MAC address in the serial monitor.
- Update the **artemis_address** in *connection.yaml*, accordingly.
- Make sure to call **ble.reload_config()** or **get_ble_controller()** (which internally calls **reload_config()**) after making any changes to your configuration file.

<hr>

In the below cell, we create an **ArtemisBLEController** object using **get_ble_controller()** (defined in *ble.py*), which creates and/or returns a single instance of **ArtemisBLEController**. <br>
<span style="color:rgb(240,50,50)"> __NOTE__: Do not use the class directly to instantiate an object. </span><br>

In [4]:
# Get ArtemisBLEController object
ble = get_ble_controller()

# Connect to the Artemis Device
ble.connect()

2026-02-04 11:34:45,932 | INFO     |: Looking for Artemis Nano Peripheral Device: c0:42:0c:78:b8:49
2026-02-04 11:34:45,932 | INFO     |: Scanning for device with address: c0:42:0c:78:b8:49, service UUID: d427e7cc-c400-4597-b417-d564e20d6600
2026-02-04 11:34:55,961 | INFO     |: Found 1 device(s) advertising service d427e7cc-c400-4597-b417-d564e20d6600
2026-02-04 11:34:55,962 | INFO     |: Selecting device: F32F03E1-EF13-6043-59E7-D2AE8E670FC2 (name: Artemis BLE)
2026-02-04 11:34:57,633 | INFO     |: Connected to c0:42:0c:78:b8:49


In [3]:
ble.reload_config()

## Receive data from the Artemis board

The cell below shows examples of reading different types (as defined in the Arduino sketch) of GATT characteristics.

In [5]:
# Read a float GATT Charactersistic
f = ble.receive_float(ble.uuid['RX_FLOAT'])
print(f)

5.0


In [6]:
# Read a string GATT Charactersistic
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

[->9.000<-]


## Send a command to the Artemis board
Send the PING command and read the reply string from the string characteristic RX_STRING. <br>
__NOTE__: The **send_command()** essentially sends a string data to the GATT characteristic (TX_CMD_STRING). The GATT characteristic in the Arduino sketch is of type BLECStringCharacteristic.

In [6]:
ble.send_command(CMD.PING, "")

In [7]:
s = ble.receive_string(ble.uuid['RX_STRING'])
print(s)

PONG


The cell below shows an example of the SEND_TWO_INTS command. <br> The two values in the **data** are separated by a delimiter "|". <br>
Refer Lab 2 documentation for more information on the command protocol.

In [8]:
ble.send_command(CMD.SEND_TWO_INTS, "2|-6")

The Artemis board should print the two integers to the serial monitor in the ArduinoIDE. 

## Disconnect

In [56]:
# Disconnect
ble.disconnect()

## Task 1
Send a string value from the computer to the Artemis board using the ECHO command. The computer should then receive and print an augmented string.

In [41]:
ble.send_command(CMD.ECHO, "HiHello")

## Task 2
Send three floats to the Artemis board using the SEND_THREE_FLOATS command and extract the three float values in the Arduino sketch.

In [42]:
ble.send_command(CMD.SEND_THREE_FLOATS, "0.1|0.2|0.25")

## Task 3
Add a command GET_TIME_MILLIS which makes the robot reply write a string such as “T:123456” to the string characteristic.

In [11]:
ble.send_command(CMD.GET_TIME_MILLIS,"")

In [12]:
ble.receive_string(ble.uuid["RX_STRING"])

'T: 38731'

## Task 4
Setup a notification handler in Python to receive the string value (the BLEStringCharactersitic in Arduino) from the Artemis board. In the callback function, extract the time from the string.

In [43]:
timestamps = []
def time_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data)
    if s.startswith("T:"):
        t = int(s[2:])
        timestamps.append(t)
        print("Time(ms) =", t)

In [44]:
ble.start_notify(ble.uuid["RX_STRING"], time_notification_handler)

In [45]:
ble.send_command(CMD.GET_TIME_MILLIS, "")

Time(ms) = 2458139


In [25]:
ble.stop_notify(ble.uuid["RX_STRING"])

## Task 5
Write a loop that gets the current time in milliseconds and sends it to your laptop to be received and processed by the notification handler. Collect these values for a few seconds and use the time stamps to determine how fast messages can be sent. What is the effective data transfer rate of this method?

In [21]:
timestamps = []
duration = 2.0
start = time.time()

while time.time() - start < duration:
    ble.send_command(CMD.GET_TIME_MILLIS, "")

In [22]:
rate = (len(timestamps))/ duration
print ("Transfer rate (msg/s)", rate)
print ("Sending Time (s/msg)", 1/rate)

Transfer rate (msg/s) 17.0
Sending Time (s/msg) 0.058823529411764705


## Task 6
Now create an array that can store time stamps. This array should be defined globally so that other functions can access it if need be. In the loop, rather than send each time stamp, place each time stamp into the array. (Note: you’ll need some extra logic to determine when your array is full so you don’t “over fill” the array.) Then add a command SEND_TIME_DATA which loops the array and sends each data point as a string to your laptop to be processed. (You can store these values in a list in python to determine if all the data was sent over.)

In [23]:
timestamps = []
ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(duration)
ble.send_command(CMD.SEND_TIME_DATA, "")

In [24]:
rate = (len(timestamps))/ duration
print ("Transfer rate (msg/s)", rate)
print ("Sending Time (s/msg)", 1/rate)

Transfer rate (msg/s) 149.5
Sending Time (s/msg) 0.006688963210702341


## ICM Sensor

In [34]:
times = []
accX = []
accY = []
accZ = []

def accl_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data).split("|")
    if len(s) == 3:
        x = float(s[0][2:])
        y = float(s[1][2:])
        z = float(s[2][2:])
        accX.append(x)
        accY.append(y)
        accZ.append(z)
        print ("X:", x, ", Y:", y, "Z:", z)

In [26]:
ble.start_notify(ble.uuid["RX_STRING"], accl_notification_handler)

In [35]:
times = []
accX = []
accY = []
accZ = []

ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(2)
ble.send_command(CMD.GET_ACCL_READINGS, "")

X: -9.765 , Y: -1009.765 Z: 4.394
X: 4.882 , Y: -1014.648 Z: 4.882
X: 0.488 , Y: -1013.183 Z: 15.625
X: -6.347 , Y: -1018.554 Z: -3.906
X: -7.324 , Y: -1019.042 Z: 18.066
X: -6.835 , Y: -1028.32 Z: -0.488
X: 3.417 , Y: -1024.902 Z: 10.253
X: 1.953 , Y: -1026.367 Z: 19.531
X: 1.464 , Y: -1009.277 Z: 10.742
X: 1.953 , Y: -1032.226 Z: -0.488
X: 1.953 , Y: -1001.953 Z: 3.417
X: 8.789 , Y: -1019.042 Z: 11.23
X: 7.324 , Y: -1011.23 Z: 6.835
X: 2.929 , Y: -1002.441 Z: 11.718
X: 8.789 , Y: -1005.371 Z: 3.417
X: 4.882 , Y: -1014.16 Z: -3.417
X: 7.324 , Y: -1005.859 Z: 8.3
X: 7.812 , Y: -1010.742 Z: 13.671
X: 9.277 , Y: -1004.394 Z: 10.253
X: -8.3 , Y: -1019.531 Z: 10.742
X: -14.16 , Y: -1008.789 Z: 1.464
X: 10.742 , Y: -1011.718 Z: 2.441
X: -3.906 , Y: -1012.695 Z: 5.859
X: 1.464 , Y: -1021.484 Z: -0.976
X: -1.464 , Y: -1017.089 Z: -8.789
X: 3.417 , Y: -1020.507 Z: 7.812
X: 5.859 , Y: -1016.601 Z: -0.976
X: -2.929 , Y: -1014.16 Z: 11.23
X: -3.417 , Y: -1008.3 Z: -0.976
X: -6.835 , Y: -1003.906 

In [36]:
accX = np.array(accX)
accY = np.array(accY)
accZ = np.array(accZ)

pitch_deg = np.degrees(np.arctan2(accX, np.sqrt(accY**2 + accZ**2)))
roll_deg  = np.degrees(np.arctan2(accY, np.sqrt(accX**2 + accZ**2)))

In [37]:
for i, (p, r) in enumerate(zip(pitch_deg, roll_deg)):
    print(f"pitch={p:7.2f}°   roll={r:7.2f}°")

pitch= -65.77°   roll= -89.75°
pitch=  45.00°   roll= -89.72°
pitch=   1.79°   roll= -89.12°
pitch=-121.61°   roll= -90.22°
pitch= -22.07°   roll= -88.98°
pitch= -94.08°   roll= -90.03°
pitch=  18.43°   roll= -89.43°
pitch=   5.71°   roll= -88.91°
pitch=   7.76°   roll= -89.39°
pitch= 104.03°   roll= -90.03°
pitch=  29.75°   roll= -89.80°
pitch=  38.05°   roll= -89.37°
pitch=  46.98°   roll= -89.61°
pitch=  14.03°   roll= -89.33°
pitch=  68.75°   roll= -89.81°
pitch= 124.99°   roll= -90.19°
pitch=  41.43°   roll= -89.53°
pitch=  29.74°   roll= -89.23°
pitch=  42.14°   roll= -89.42°
pitch= -37.69°   roll= -89.40°
pitch= -84.10°   roll= -89.92°
pitch=  77.20°   roll= -89.86°
pitch= -33.69°   roll= -89.67°
pitch= 123.69°   roll= -90.05°
pitch=-170.54°   roll= -90.50°
pitch=  23.62°   roll= -89.56°
pitch=  99.46°   roll= -90.06°
pitch= -14.62°   roll= -89.37°
pitch=-105.94°   roll= -90.06°
pitch= -70.35°   roll= -89.86°
pitch=  33.69°   roll= -89.84°
pitch=  31.76°   roll= -89.43°
pitch=-1

## Task 7
Add a second array that is the same size as the time stamp array. Use this array to store temperature readings. Each element in both arrays should correspond, e.e., the first time stamp was recorded at the same time as the first temperature reading. Then add a command GET_TEMP_READINGS that loops through both arrays concurrently and sends each temperature reading with a time stamp. The notification handler should parse these strings and add populate the data into two lists.

In [31]:
times = []
temps = []

def temp_notification_handler(sender, data: bytearray):
    s = ble.bytearray_to_string(data).split("|")
    if len(s) == 2:
        time = int(s[0][2:])
        temp = float(s[1][2:])
        times.append(time)
        temps.append(temp)
        print ("T:", time, ", F:", temp)

In [32]:
ble.start_notify(ble.uuid["RX_STRING"], temp_notification_handler)

In [33]:
times = []
temps = []

ble.send_command(CMD.START_COLLECT_DATA, "")
time.sleep(duration)
ble.send_command(CMD.GET_TEMP_READINGS, "")

T: 135829 , F: 79.0
T: 135829 , F: 80.0
T: 135829 , F: 79.0
T: 135830 , F: 80.0
T: 135830 , F: 80.0
T: 135830 , F: 79.0
T: 135830 , F: 80.0
T: 135831 , F: 80.0
T: 135831 , F: 80.0
T: 135831 , F: 79.0
T: 135832 , F: 80.0
T: 135832 , F: 80.0
T: 135832 , F: 80.0
T: 135832 , F: 80.0
T: 135833 , F: 80.0
T: 135833 , F: 80.0
T: 135833 , F: 79.0
T: 135833 , F: 80.0
T: 135834 , F: 80.0
T: 135834 , F: 80.0
T: 135834 , F: 80.0
T: 135834 , F: 80.0
T: 135835 , F: 79.0
T: 135835 , F: 80.0
T: 135835 , F: 80.0
T: 135836 , F: 80.0
T: 135836 , F: 80.0
T: 135836 , F: 80.0
T: 135836 , F: 80.0
T: 135837 , F: 80.0
T: 135837 , F: 80.0
T: 135837 , F: 80.0
T: 135837 , F: 80.0
T: 135838 , F: 80.0
T: 135838 , F: 80.0
T: 135838 , F: 80.0
T: 135838 , F: 80.0
T: 135839 , F: 80.0
T: 135839 , F: 79.0
T: 135839 , F: 80.0
T: 135839 , F: 80.0
T: 135845 , F: 79.0
T: 135845 , F: 80.0
T: 135845 , F: 80.0
T: 135845 , F: 80.0
T: 135846 , F: 80.0
T: 135846 , F: 82.0
T: 135846 , F: 80.0
T: 135847 , F: 79.0
T: 135847 , F: 80.0


In [34]:
ble.stop_notify(ble.uuid["RX_STRING"])